In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import sunpy.visualization.colormaps as cm

import glob
from processing import *
from classical_estimates import classical_estimates
from fit_pv import *

In [2]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'
deadpix_file = '/home/ulyanov/data/solo/phi/dead_pixels/phi-fdt-deadpix_20250915T140003_V202609091415C_0569150100.fits'
prefilter_file = '/home/ulyanov/data/solo/phi/prefilter/phi-fdt-prefilter_20250916T023002_V202607231634C_0569160250.txt'
distortion_file = '/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz'

In [3]:
flat_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*flat*.fits'))
ghost_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*ghost*.fits'))
cavity_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*cavity*.fits'))

print(flat_files)

['/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20240330T050009_V202608262158C_0463300100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20240926T114503_V202608262134C_0469260100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20241016T113003_V202608262111C_0470160100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20241027T233003_V202608262048C_0470270100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20241202T123003_V202608262027C_0472020100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250119T210009_V202608262003C_0561190100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250310T080009_V202608261939C_0563100100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250915T140003_V202608261916C_0569150100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250923T000503_V202608261853C_0569230100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20260310T040003_V202608261828C_0663100

In [4]:
i = -2
cavity_file, flat_file, ghost_file = cavity_files[i], flat_files[i], ghost_files[i]

In [6]:
folder_blos = '/home/ulyanov/data/solo/phi/2026/blos_/'
folder_vlos = '/home/ulyanov/data/solo/phi/2026/vlos_/'

In [8]:
import fnmatch
from connect import bob

sftp = bob()

top_dir = '/data/solo/phi/data/fmdb/l1/'
dirs = sorted(sftp.listdir(top_dir))

folder = '/home/ulyanov/data/solo/phi/2026/'

for directory in dirs:
    if fnmatch.fnmatch(directory, '2026-0[5-6]*'):
        for file in sorted(sftp.listdir(top_dir + directory))[:1]:
            if fnmatch.fnmatch(file, '*fdt-alam*C_*.fits.gz'):
                print(file)

                remote_file = top_dir + directory + '/' + file
                local_file = 'temp.fits.gz'
                sftp.get(remote_file, local_file)

                data, header = process(local_file,
                                       dark_file=dark_file,
                                       deadpix_file=deadpix_file,
                                       prefilter_file=prefilter_file,
                                       #cavity_file=cavity_file,
                                       flatfield_file=flat_file,
                                       ghost_file=ghost_file,
                                       distortion_file=distortion_file,
                                       _realign=True,
                                       _find_center=True,
                                       _demodulate=True,
                                       _correct_fringes=True,
                                       _correct_crosstalk=True,
                                       _calc_wavelengths=True,
                                       _mask=True,
                                       )

                Blos, Vlos = classical_estimates(data, header, lam=1e-3, niter=10, five_points=True)

                file_blos = generate_filename(file, prefix='blos', folder=folder_blos)
                file_vlos = generate_filename(file, prefix='vlos', folder=folder_vlos)
                clone_fits(local_file, file_blos, Blos, header)
                clone_fits(local_file, file_vlos, Vlos, header)

solo_L1_phi-fdt-alam_20260501T020003_V202605261830C_0645010501.fits.gz
solo_L1_phi-fdt-alam_20260515T020003_V202606131630C_0645150501.fits.gz
solo_L1_phi-fdt-alam_20260516T020003_V202606141630C_0645160501.fits.gz
solo_L1_phi-fdt-alam_20260517T020003_V202606141930C_0645170501.fits.gz
solo_L1_phi-fdt-alam_20260518T020003_V202606151630C_0645180501.fits.gz
solo_L1_phi-fdt-alam_20260519T020003_V202606151831C_0645190501.fits.gz
solo_L1_phi-fdt-alam_20260520T013003_V202606161132C_0645200501.fits.gz
solo_L1_phi-fdt-alam_20260521T020003_V202606181531C_0645210501.fits.gz
solo_L1_phi-fdt-alam_20260522T020003_V202606181641C_0645220501.fits.gz
solo_L1_phi-fdt-alam_20260523T020003_V202606181732C_0645230501.fits.gz
solo_L1_phi-fdt-alam_20260524T020003_V202606181832C_0645240501.fits.gz
solo_L1_phi-fdt-alam_20260525T020003_V202606181932C_0645250501.fits.gz
solo_L1_phi-fdt-alam_20260526T050003_V202606200932C_0645260501.fits.gz
solo_L1_phi-fdt-alam_20260527T020009_V202606200836C_0645270501.fits.gz
solo_L

KeyboardInterrupt: 

In [8]:
generate_filename(file, prefix='blos', folder=folder_blos)

'/home/ulyanov/data/solo/phi/2026/blos/phi-fdt-blos_20250101T001503_V202609101723C_0541011501.fits'